In [8]:
# ---------------------- TASK 1 ---------------------- #

import random
suits = ['Hearts', 'Diamonds', 'Clubs', 'Spades']
ranks = ['2', '3', '4', '5', '6', '7', '8', '9', '10', 'Jack', 'Queen', 'King', 'Ace']
deck = [(rank, suit) for suit in suits for rank in ranks]

#1 Probability of drawing a red card (Hearts or Diamonds)
red_cards = [card for card in deck if card[1] in ['Hearts', 'Diamonds']]
prob_red_theoratical = len(red_cards) / len(deck)

#2 Given red card, probability of it being a Heart
hearts = [card for card in red_cards if card[1] == 'Hearts']
prob_heart_given_red_theoratical = len(hearts) / len(red_cards)


#3 Given face card, probability of it being a Diamond
face_cards = [card for card in deck if card[0] in ['Jack', 'Queen', 'King']]
diamond_face = [card for card in face_cards if card[1] == 'Diamonds']
prob_diamond_given_face_theoratical = len(diamond_face) / len(face_cards)

#4 Given face card, probability of it's a spade or a queen
spade_face = [card for card in face_cards if card[1] == 'Spades']
queens = [card for card in face_cards if card[0] == 'Queen']
spade_or_queen = set(spade_face) | set(queens)
prob_spade_or_queen_given_face_theoratical = len(spade_or_queen) / len(face_cards)



trials = 100000
red_count = 0
heart_given_red_count = 0
face_count = 0
diamond_given_face_count = 0
spade_or_queen_given_face_count = 0

for _ in range(trials):
    card = random.choice(deck)
    # Red card
    if card in red_cards:
        red_count += 1
        if card in hearts:
            heart_given_red_count += 1
    # Face card
    if card in face_cards:
        face_count += 1
        if card in diamond_face:
            diamond_given_face_count += 1
        if card in spade_or_queen:
            spade_or_queen_given_face_count += 1

prob_red = red_count / trials
prob_heart_given_red = heart_given_red_count / red_count if red_count > 0 else 0
prob_diamond_given_face = diamond_given_face_count / face_count if face_count > 0 else 0
prob_spade_or_queen_given_face = spade_or_queen_given_face_count / face_count if face_count > 0 else 0

print(f"Theoretical P(Red) = {prob_red_theoratical:.3f}, Simulated P(Red) = {prob_red:.3f}")
print(f"Theoretical P(Heart|Red) = {prob_heart_given_red_theoratical:.3f}, Simulated P(Heart|Red) = {prob_heart_given_red:.3f}")
print(f"Theoretical P(Diamond|Face) = {prob_diamond_given_face_theoratical:.3f}, Simulated P(Diamond|Face) = {prob_diamond_given_face:.3f}")
print(f"Theoretical P(Spade or Queen|Face) = {prob_spade_or_queen_given_face_theoratical:.3f}, Simulated P(Spade or Queen|Face) = {prob_spade_or_queen_given_face:.3f}")






Theoretical P(Red) = 0.500, Simulated P(Red) = 0.501
Theoretical P(Heart|Red) = 0.500, Simulated P(Heart|Red) = 0.501
Theoretical P(Diamond|Face) = 0.250, Simulated P(Diamond|Face) = 0.245
Theoretical P(Spade or Queen|Face) = 0.500, Simulated P(Spade or Queen|Face) = 0.503


In [9]:
# ---------------------- TASK 2 ---------------------- #

from pgmpy.models import DiscreteBayesianNetwork
from pgmpy.factors.discrete import TabularCPD
from pgmpy.inference import VariableElimination

model = DiscreteBayesianNetwork([
    ('Intelligence', 'Grade'),
    ('StudyHours', 'Grade'),
    ('Difficulty', 'Grade'),
    ('Grade', 'Pass')
])

cpd_intelligence = TabularCPD(
    variable='Intelligence', 
    variable_card=2,
    values=[[0.7], [0.3]],
    state_names={'Intelligence': ['High', 'Low']}
)

cpd_StudyHours = TabularCPD(
    variable='StudyHours', 
    variable_card=2,
    values=[[0.6], [0.4]],
    state_names={'StudyHours': ['Sufficient', 'Insufficient']}
)

cpd_difficulty = TabularCPD(
    variable='Difficulty', 
    variable_card=2,
    values=[[0.4], [0.6]],
    state_names={'Difficulty': ['Hard', 'Easy']}
)

cpd_grade = TabularCPD(
	variable='Grade',
	variable_card=3,
	values=[
		# A
		[0.7, 0.8, 0.5, 0.6, 0.5, 0.6, 0.3, 0.4],
		# B
		[0.2, 0.15, 0.3, 0.25, 0.3, 0.25, 0.4, 0.4],
		# C
		[0.1, 0.05, 0.2, 0.15, 0.2, 0.15, 0.3, 0.2],
	],
	evidence=['Intelligence', 'StudyHours', 'Difficulty'],
	evidence_card=[2, 2, 2],
	state_names={
		'Grade': ['A', 'B', 'C'],
		'Intelligence': ['High', 'Low'],
		'StudyHours': ['Sufficient', 'Insufficient'],
		'Difficulty': ['Hard', 'Easy']
	}
)

cpd_pass = TabularCPD(
    variable='Pass',
    variable_card=2,
    values=[
        [0.95, 0.8, 0.5],
        [0.05, 0.2, 0.5]
    ],
    evidence=['Grade'],
    evidence_card=[3],
    state_names={
        'Pass': ['Yes', 'No'],
        'Grade': ['A', 'B', 'C']
    }
)

model.add_cpds(cpd_intelligence, cpd_StudyHours, cpd_difficulty, cpd_grade, cpd_pass)

assert model.check_model()
inference = VariableElimination(model)

# Query 1: P(Pass | StudyHours=Sufficient, Difficulty=Hard)
result1 = inference.query(
    variables=['Pass'],
    evidence={'StudyHours': 'Sufficient', 'Difficulty': 'Hard'}
)

print(f"P(Pass | StudyHours=Sufficient, Difficulty=Hard):")
print(result1)

# Query 2: P(Intelligence=High | Pass=Yes)
result2 = inference.query(
    variables=['Intelligence'],
    evidence={'Pass': 'Yes'}
)
print(f"P(Intelligence=High | Pass=Yes):")
print(result2)






P(Pass | StudyHours=Sufficient, Difficulty=Hard):
+-----------+-------------+
| Pass      |   phi(Pass) |
+===========+=============+
| Pass(Yes) |      0.8570 |
+-----------+-------------+
| Pass(No)  |      0.1430 |
+-----------+-------------+
P(Intelligence=High | Pass=Yes):
+--------------------+---------------------+
| Intelligence       |   phi(Intelligence) |
+====================+=====================+
| Intelligence(High) |              0.7139 |
+--------------------+---------------------+
| Intelligence(Low)  |              0.2861 |
+--------------------+---------------------+


In [10]:
# ---------------------- TASK 3 ---------------------- #



from pgmpy.models import DiscreteBayesianNetwork
from pgmpy.factors.discrete import TabularCPD
from pgmpy.inference import VariableElimination

model = DiscreteBayesianNetwork([
	('Disease', 'Fever'),
	('Disease', 'Cough'),
	('Disease', 'Fatigue'),
	('Disease', 'Chills')
])

cpd_disease = TabularCPD(
	variable='Disease',
	variable_card=2,
	values=[[0.3], [0.7]],  # [Flu, Cold]
	state_names={'Disease': ['Flu', 'Cold']}
)

cpd_fever = TabularCPD(
	variable='Fever',
	variable_card=2,
	values=[
		[0.9, 0.5],
		[0.1, 0.5]
	],
	evidence=['Disease'],
	evidence_card=[2],
	state_names={
		'Fever': ['Yes', 'No'],
		'Disease': ['Flu', 'Cold']
	}
)

cpd_cough = TabularCPD(
	variable='Cough',
	variable_card=2,
	values=[
		[0.8, 0.6],
		[0.2, 0.4]
	],
	evidence=['Disease'],
	evidence_card=[2],
	state_names={
		'Cough': ['Yes', 'No'],
		'Disease': ['Flu', 'Cold']
	}
)

cpd_fatigue = TabularCPD(
	variable='Fatigue',
	variable_card=2,
	values=[
		[0.7, 0.3],
		[0.3, 0.7]
	],
	evidence=['Disease'],
	evidence_card=[2],
	state_names={
		'Fatigue': ['Yes', 'No'],
		'Disease': ['Flu', 'Cold']
	}
)

cpd_chills = TabularCPD(
	variable='Chills',
	variable_card=2,
	values=[
		[0.6, 0.4],
		[0.4, 0.6]
	],
	evidence=['Disease'],
	evidence_card=[2],
	state_names={
		'Chills': ['Yes', 'No'],
		'Disease': ['Flu', 'Cold']
	}
)

model.add_cpds(cpd_disease, cpd_fever, cpd_cough, cpd_fatigue, cpd_chills)

assert model.check_model(), "Model is incorrect"

infer = VariableElimination(model)

# --- Inference Task 1 ---
# P(Disease | Fever=Yes, Cough=Yes)
result1 = infer.query(
	variables=['Disease'],
	evidence={'Fever': 'Yes', 'Cough': 'Yes'}
)
print("Task 1: P(Disease | Fever=Yes, Cough=Yes)")
print(result1)

# --- Inference Task 2 ---
# P(Disease | Fever=Yes, Cough=Yes, Chills=Yes)
result2 = infer.query(
	variables=['Disease'],
	evidence={'Fever': 'Yes', 'Cough': 'Yes', 'Chills': 'Yes'}
)
print("\nTask 2: P(Disease | Fever=Yes, Cough=Yes, Chills=Yes)")
print(result2)

# --- Inference Task 3 ---
# P(Fatigue=Yes | Disease=Flu)
result3 = infer.query(
	variables=['Fatigue'],
	evidence={'Disease': 'Flu'}
)
print("\nTask 3: P(Fatigue=Yes | Disease=Flu)")
print(f"P(Fatigue=Yes | Disease=Flu): {result3.values[0]:.2f}")


Task 1: P(Disease | Fever=Yes, Cough=Yes)
+---------------+----------------+
| Disease       |   phi(Disease) |
+===============+================+
| Disease(Flu)  |         0.5070 |
+---------------+----------------+
| Disease(Cold) |         0.4930 |
+---------------+----------------+

Task 2: P(Disease | Fever=Yes, Cough=Yes, Chills=Yes)
+---------------+----------------+
| Disease       |   phi(Disease) |
+===============+================+
| Disease(Flu)  |         0.6067 |
+---------------+----------------+
| Disease(Cold) |         0.3933 |
+---------------+----------------+

Task 3: P(Fatigue=Yes | Disease=Flu)
P(Fatigue=Yes | Disease=Flu): 0.70


In [11]:
# ---------------------- TASK 4 ---------------------- #

import numpy as np

states = ["Sunny", "Cloudy", "Rainy"]
state_to_idx = {state: i for i, state in enumerate(states)}

transition_matrix = np.array([
	[0.7, 0.2, 0.1],  # Sunny -> Sunny, Cloudy, Rainy
	[0.3, 0.4, 0.3],  # Cloudy -> Sunny, Cloudy, Rainy
	[0.2, 0.3, 0.5],  # Rainy -> Sunny, Cloudy, Rainy
])

def simulate_weather(start_state, days, transition_matrix):
	current_state = state_to_idx[start_state]
	weather_sequence = [current_state]
	for _ in range(days - 1):
		current_state = np.random.choice(
			[0, 1, 2], p=transition_matrix[current_state]
		)
		weather_sequence.append(current_state)
	return weather_sequence

np.random.seed(42)
simulated_sequence = simulate_weather("Sunny", 10, transition_matrix)
simulated_weather = [states[i] for i in simulated_sequence]
print("Simulated weather for 10 days:", simulated_weather)

def estimate_rainy_probability(trials=10000):
	count = 0
	for _ in range(trials):
		seq = simulate_weather("Sunny", 10, transition_matrix)
		rainy_days = sum(1 for s in seq if s == state_to_idx["Rainy"])
		if rainy_days >= 3:
			count += 1
	return count / trials

prob = estimate_rainy_probability()
print(f"Probability of at least 3 rainy days in 10 days: {prob:.4f}")


Simulated weather for 10 days: ['Sunny', 'Sunny', 'Rainy', 'Rainy', 'Rainy', 'Sunny', 'Sunny', 'Sunny', 'Cloudy', 'Cloudy']
Probability of at least 3 rainy days in 10 days: 0.3472
